<a href="https://colab.research.google.com/github/getuliojql/SISSA-CIN0114/blob/main/projeto/notebooks/02_dataset_kim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim"

Mounted at /content/drive


In [2]:
import os, sys, time
import numpy as np
import pandas as pd

PASTA = '/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim'
SAIDA = os.path.join(PASTA, 'parsed')
ARQUIVOS = ['benign_traffic.pcap', 'dos_noti_flood.pcap', 'fuzzy_sd_offer_rand_noti(1).pcap',
            'fuzzy_sd_offer_rand_noti(2).pcap', 'fuzzy_sd_offer_rand_noti(3).pcap',
            'mitm_multi_attacker.pcap', 'mitm_single_attacker.pcap']

In [2]:
import requests
arquivos = requests.get('https://api.figshare.com/v2/articles/30970450/files').json()
for a in arquivos:
  print(a['name'], round(a['size'] / 1e6), 'MB', a['download_url'])

benign_traffic.pcap 223 MB https://ndownloader.figshare.com/files/61471573
dos_noti_flood.pcap 195 MB https://ndownloader.figshare.com/files/61471570
fuzzy_sd_offer_rand_noti(1).pcap 230 MB https://ndownloader.figshare.com/files/61471576
fuzzy_sd_offer_rand_noti(2).pcap 133 MB https://ndownloader.figshare.com/files/61471567
fuzzy_sd_offer_rand_noti(3).pcap 226 MB https://ndownloader.figshare.com/files/61471579
mitm_multi_attacker.pcap 250 MB https://ndownloader.figshare.com/files/61471585
mitm_single_attacker.pcap 214 MB https://ndownloader.figshare.com/files/61471582
tr_te_sets.tar 797 MB https://ndownloader.figshare.com/files/61471600


In [3]:
import os
DEST = '/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim'
for a in arquivos:
  caminho = os.path.join(DEST, a['name'])
  if not os.path.exists(caminho):
    !wget -q -O "{caminho}" "{a['download_url']}"

In [4]:
!tar -tvf "/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/tr_te_sets.tar" | head -50

-rw-rw-rw- 0/0        56933520 2026-01-29 15:23 y_train.npy
-rw-rw-rw- 0/0        56933512 2026-01-29 15:23 y_test.npy
-rw-rw-rw- 0/0       341600480 2026-01-29 15:24 X_train.npy
-rw-rw-rw- 0/0       341600432 2026-01-29 15:24 X_test.npy


In [5]:
!mkdir -p /content/kim_tr_te
!tar -xf "/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/tr_te_sets.tar" -C /content/kim_tr_te
!find /content/kim_tr_te -type f -exec ls -lh {} \; | head -50
!du -sh /content/kim_tr_te

-rw-rw-rw- 1 root root 326M Jan 29  2026 /content/kim_tr_te/X_train.npy
-rw-rw-rw- 1 root root 55M Jan 29  2026 /content/kim_tr_te/y_test.npy
-rw-rw-rw- 1 root root 55M Jan 29  2026 /content/kim_tr_te/y_train.npy
-rw-rw-rw- 1 root root 326M Jan 29  2026 /content/kim_tr_te/X_test.npy
761M	/content/kim_tr_te


In [7]:
import numpy as np
np.set_printoptions(suppress=True, linewidth=200, precision=4)
BASE = '/content/kim_tr_te'

# 1) Formato e tipo de cada array (mmap_mode='r' faz leitura sob demanda, sem carregar o arquivo inteiro na memória)
for nome in ['X_train', 'y_train', 'X_test', 'y_test']:
  a = np.load(f'{BASE}/{nome}.npy', mmap_mode='r')
  print(f'{nome}: shape={a.shape}, dtype={a.dtype}')

X_tr = np.load(f'{BASE}/X_train.npy', mmap_mode='r')
y_tr = np.load(f'{BASE}/y_train.npy', mmap_mode='r')
y_te = np.load(f'{BASE}/y_test.npy', mmap_mode='r')

# 2) Rótulos presentes em cada conjunto e respectivas contagens
for nome, y in [('treino', y_tr), ('teste', y_te)]:
  valores, contagens = np.unique(y, return_counts=True)
  print(f'\nRótulos no {nome}:', dict(zip(valores.tolist(), contagens.tolist())))

# 3) Exibe as primeiras linhas de X_train e de y_train para inspeção dos atributos e rótulos
print('\nPrimeiras 10 linhas de X_train:\n', np.asarray(X_tr[:10]))
print('Primeiros 30 rótulos de y_train:', np.asarray(y_tr[:30]))

# 4) Estatísticas por coluna (mínimo, máximo e número de valores distintos) em uma amostra de 200 mil linhas
amostra = np.asarray(X_tr[:200_000])
print('\ncoluna | mínimo | máximo | valores distintos')
for j in range(amostra.shape[1]):
  col = amostra[:, j]
  print(f'{j:6} | {col.min():.4f} | {col.max():.4f} | {len(np.unique(col))}')

# 5) Verificação de ordem temporal: em dados ordenados, o rótulo varia pouco entre linhas consecutivas
trecho = np.asarray(y_tr[:1_000_000])
print('\nMudanças de rótulo entre linhas vizinhas (no primeiro milhão):',
      np.count_nonzero(np.diff(trecho) != 0))

X_train: shape=(7116674, 12), dtype=float32
y_train: shape=(7116674,), dtype=int64
X_test: shape=(7116673, 12), dtype=float32
y_test: shape=(7116673,), dtype=int64

Rótulos no treino: {0: 6285515, 1: 831159}

Rótulos no teste: {0: 6285514, 1: 831159}

Primeiras 10 linhas de X_train:
 [[0.4777 0.5223 0.6676 1.     0.3324 0.     0.     0.5    0.8828 0.2535 0.0009 0.    ]
 [0.7306 0.2694 0.843  1.     0.157  0.     0.     0.494  0.8813 0.6236 0.0499 0.0312]
 [0.7306 0.2694 0.843  1.     0.157  0.     0.     0.488  0.8799 0.2142 0.0664 0.0639]
 [0.4777 0.5223 0.6676 1.     0.3324 0.     0.     0.494  0.8813 0.0914 0.0517 0.0345]
 [0.4007 0.5993 0.4634 1.     0.5366 0.     0.     0.512  0.8856 0.043  0.0673 0.0639]
 [0.4736 0.5264 0.6646 1.     0.3354 0.     0.     0.5    0.8828 0.2539 0.0009 0.    ]
 [0.4736 0.5264 0.6646 1.     0.3354 0.     0.     0.506  0.8842 0.121  0.0517 0.0322]
 [0.4007 0.5993 0.4634 1.     0.5366 0.     0.     0.512  0.8856 0.0427 0.0664 0.0639]
 [0.7306 0.2694 0.8

In [9]:
# 6) Estrutura dos rótulos ao longo dos arrays completos
for nome, y in [('treino', y_tr), ('teste', y_te)]:
  y_mem = np.asarray(y)
  mudancas = np.flatnonzero(np.diff(y_mem) != 0) + 1
  print(f'\n{nome}: {len(mudancas)} mudanças de rótulo')
  print('  primeiras posições de mudança:', mudancas[:10].tolist())
  print('  últimas posições de mudança:', mudancas[-10:].tolist())
  print('  primeiro índice com rótulo 1:', int(np.argmax(y_mem == 1)))

# 7) Instalação do tshark (inclui o capinfos) e contagem de pacotes de cada pcap
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y tshark > /dev/null
!capinfos -c "/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/"*.pcap


treino: 2 mudanças de rótulo
  primeiras posições de mudança: [1096901, 1928060]
  últimas posições de mudança: [1096901, 1928060]
  primeiro índice com rótulo 1: 1096901

teste: 2 mudanças de rótulo
  primeiras posições de mudança: [1096900, 1928059]
  últimas posições de mudança: [1096900, 1928059]
  primeiro índice com rótulo 1: 1096900
File name:           /content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/benign_traffic.pcap
Number of packets:   2,194 k

File name:           /content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/dos_noti_flood.pcap
Number of packets:   1,931 k

File name:           /content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/fuzzy_sd_offer_rand_noti(1).pcap
Number of packets:   2,198 k

File name:           /content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/fuzzy_sd_offer_rand_noti(2).pcap
Number of packets:   1,305 k

File name:           /content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim/fuzzy_sd_offer_rand_noti(3).pcap
Number of packets:

In [12]:
# 8) Download do dataset original do SISSA para consulta do formato de entrada (data/raw/data.csv)
!gdown -q 'https://drive.google.com/uc?id=1-pl9OOFcTZCuPLRzusjlmkMn7Q-f-0fU' -O /content/sissa_data.7z
!7z x -y -o/content/sissa_data /content/sissa_data.7z > /dev/null

import pandas as pd
raw = pd.read_csv('/content/sissa_data/data/raw/data.csv', nrows=20)
print('Colunas:', raw.columns.tolist())
print(raw.head(10).to_string())

Colunas: ['time', 'src', 'dst', 'method', 'length', 'client', 'session', 'protocol', 'interface', 'message_type', 'err_code']
           time       src       dst  method  length  client  session  protocol  interface  message_type     err_code
0  1.700214e+09  10.1.0.8  10.0.0.2       1       8       8        1         1          1       REQUEST         E_OK
1  1.700214e+09  10.1.0.5  10.0.0.1     273      14       5        1         1          1       REQUEST         E_OK
2  1.700214e+09  10.1.0.2  10.0.0.2       1      13       2        1         1          1       REQUEST         E_OK
3  1.700214e+09  10.1.0.1  10.0.0.1     273      25       1        1         1          1       REQUEST         E_OK
4  1.700214e+09  10.0.0.1  10.1.0.5     273      26       5        1         1          1      RESPONSE         E_OK
5  1.700214e+09  10.1.0.3  10.0.0.2       1      24       3        1         1          1       REQUEST         E_OK
6  1.700214e+09  10.1.0.8  10.0.0.2       1      22    

In [13]:
# 9) Validação da regra de rotulagem por nó atacante: pacotes cujo IP de origem não ocorre no tráfego benigno
import subprocess, collections, os
PASTA = '/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim'

def ips_origem(arquivo):
    """Retorna a lista de IPs de origem de todos os pacotes IPv4 do pcap informado."""
    saida = subprocess.run(['tshark', '-n', '-r', os.path.join(PASTA, arquivo), '-T', 'fields', '-e', 'ip.src'],
                           capture_output=True, text=True, check=True).stdout
    return [ip for ip in saida.split('\n') if ip]

ips_benignos = set(ips_origem('benign_traffic.pcap'))
print('IPs de origem no tráfego benigno:', sorted(ips_benignos))

arquivos_ataque = ['dos_noti_flood.pcap', 'fuzzy_sd_offer_rand_noti(1).pcap', 'fuzzy_sd_offer_rand_noti(2).pcap',
                   'fuzzy_sd_offer_rand_noti(3).pcap', 'mitm_multi_attacker.pcap', 'mitm_single_attacker.pcap']
total_ataque = 0
for arquivo in arquivos_ataque:
    ips = ips_origem(arquivo)
    externos = collections.Counter(ip for ip in ips if ip not in ips_benignos)
    total_ataque += sum(externos.values())
    print(f'{arquivo}: {len(ips)} pacotes IPv4; {sum(externos.values())} de IPs ausentes no benigno {dict(externos)}')

print(f'\nTotal de pacotes de ataque pela regra: {total_ataque} | referência (Kim et al.): 1662318')

IPs de origem no tráfego benigno: ['172.18.0.10', '172.18.0.2', '172.18.0.3', '172.18.0.4', '172.18.0.5', '172.18.0.6', '172.18.0.7', '172.18.0.8', '172.18.0.9']
dos_noti_flood.pcap: 1930421 pacotes IPv4; 372817 de IPs ausentes no benigno {'172.18.0.11': 372817}
fuzzy_sd_offer_rand_noti(1).pcap: 2197133 pacotes IPv4; 5245 de IPs ausentes no benigno {'172.18.0.17': 5245}
fuzzy_sd_offer_rand_noti(2).pcap: 1304178 pacotes IPv4; 43215 de IPs ausentes no benigno {'172.18.0.12': 43215}
fuzzy_sd_offer_rand_noti(3).pcap: 2223670 pacotes IPv4; 511600 de IPs ausentes no benigno {'172.18.0.12': 511600}
mitm_multi_attacker.pcap: 2412560 pacotes IPv4; 427506 de IPs ausentes no benigno {'172.18.0.14': 258841, '172.18.0.15': 168656, '172.18.0.2,172.18.0.10': 9}
mitm_single_attacker.pcap: 2105920 pacotes IPv4; 428836 de IPs ausentes no benigno {'172.18.0.13': 428836}

Total de pacotes de ataque pela regra: 1789219 | referência (Kim et al.): 1662318


In [14]:
# 10) Composição dos pacotes do nó atacante por protocolo de transporte e por fragmentação IP
def composicao(arquivo, ip_atacante):
    """Conta os pacotes de um IP de origem por protocolo (ip.proto) e quantos são fragmentos IP."""
    saida = subprocess.run(['tshark', '-n', '-r', os.path.join(PASTA, arquivo), '-Y', f'ip.src == {ip_atacante}',
                            '-T', 'fields', '-e', 'ip.proto', '-e', 'ip.flags.mf', '-e', 'ip.frag_offset'],
                           capture_output=True, text=True, check=True).stdout
    linhas = [l.split('\t') for l in saida.split('\n') if l]
    por_protocolo = collections.Counter(l[0] for l in linhas)
    fragmentos = sum(1 for l in linhas if l[1] in ('1', 'True') or l[2] not in ('', '0'))
    return por_protocolo, fragmentos

for arquivo, ip in [('dos_noti_flood.pcap', '172.18.0.11'), ('mitm_single_attacker.pcap', '172.18.0.13')]:
    por_protocolo, fragmentos = composicao(arquivo, ip)
    print(f'{arquivo} ({ip}): por protocolo {dict(por_protocolo)}; fragmentos IP: {fragmentos}')

dos_noti_flood.pcap (172.18.0.11): por protocolo {'2': 65873, '17': 182467, '6': 124477}; fragmentos IP: 0
mitm_single_attacker.pcap (172.18.0.13): por protocolo {'17': 306886, '2': 68326, '6': 53624}; fragmentos IP: 0


In [15]:
# 11) Extração das mensagens SOME/IP de cada pcap e gravação em formato parquet no Drive
import os, sys, time
import pandas as pd

PASTA = '/content/drive/MyDrive/2026.2/cin0114 - tadi/SISSA_kim'
SAIDA = os.path.join(PASTA, 'parsed')
ARQUIVOS = ['benign_traffic.pcap', 'dos_noti_flood.pcap', 'fuzzy_sd_offer_rand_noti(1).pcap',
            'fuzzy_sd_offer_rand_noti(2).pcap', 'fuzzy_sd_offer_rand_noti(3).pcap',
            'mitm_multi_attacker.pcap', 'mitm_single_attacker.pcap']

# Obtém o parser a partir do repositório do grupo
if not os.path.exists('/content/SISSA'):
    !git clone -q https://github.com/getuliojql/SISSA-CIN0114.git /content/SISSA
sys.path.insert(0, '/content/SISSA/projeto/kim')
from parse_pcap import parse_pcap

os.makedirs(SAIDA, exist_ok=True)
estatisticas = {}
for arquivo in ARQUIVOS:
    inicio = time.time()
    tabela, stats = parse_pcap(os.path.join(PASTA, arquivo))
    tabela.to_parquet(os.path.join(SAIDA, arquivo.replace('.pcap', '.parquet')), index=False)
    estatisticas[arquivo] = stats
    print(f'{arquivo}: {stats["pacotes"]} pacotes, {stats["mensagens"]} mensagens SOME/IP '
          f'({time.time() - inicio:.0f} s)')

# Tabela de contagens por arquivo: mensagens extraídas e motivos de descarte
resumo = pd.DataFrame(estatisticas).fillna(0).astype(int)
print('\n', resumo.to_string())

benign_traffic.pcap: 2194700 pacotes, 1108704 mensagens SOME/IP (20 s)
dos_noti_flood.pcap: 1931593 pacotes, 973423 mensagens SOME/IP (19 s)
fuzzy_sd_offer_rand_noti(1).pcap: 2198015 pacotes, 1113682 mensagens SOME/IP (17 s)
fuzzy_sd_offer_rand_noti(2).pcap: 1305398 pacotes, 652722 mensagens SOME/IP (10 s)
fuzzy_sd_offer_rand_noti(3).pcap: 2224634 pacotes, 1123430 mensagens SOME/IP (18 s)
mitm_multi_attacker.pcap: 2413810 pacotes, 1293930 mensagens SOME/IP (19 s)
mitm_single_attacker.pcap: 2107242 pacotes, 1131971 mensagens SOME/IP (15 s)

                          benign_traffic.pcap  dos_noti_flood.pcap  fuzzy_sd_offer_rand_noti(1).pcap  fuzzy_sd_offer_rand_noti(2).pcap  fuzzy_sd_offer_rand_noti(3).pcap  mitm_multi_attacker.pcap  mitm_single_attacker.pcap
pacotes                              2194700              1931593                           2198015                           1305398                           2224634                   2413810                    2107242
descartados

In [18]:
# 12) Rotulagem por nó atacante e contagem de pedaços de 512 mensagens por classe
IPS_BENIGNOS = {f'172.18.0.{i}' for i in range(2, 11)}
CLASSES = {
    'benign_traffic.pcap': 'Normal',
    'dos_noti_flood.pcap': 'DoS',
    'fuzzy_sd_offer_rand_noti(1).pcap': 'Fuzzy',
    'fuzzy_sd_offer_rand_noti(2).pcap': 'Fuzzy',
    'fuzzy_sd_offer_rand_noti(3).pcap': 'Fuzzy',
    'mitm_multi_attacker.pcap': 'MITM_multi',
    'mitm_single_attacker.pcap': 'MITM_single',
}

linhas = []
for arquivo in ARQUIVOS:
    tabela = pd.read_parquet(os.path.join(SAIDA, arquivo.replace('.pcap', '.parquet')), columns=['src'])
    classe = CLASSES[arquivo]
    # Mensagem do atacante: arquivo de ataque e IP de origem fora do conjunto benigno
    atacante = (~tabela['src'].isin(IPS_BENIGNOS)).to_numpy() if classe != 'Normal' else np.zeros(len(tabela), bool)
    # Pedaços de 512 mensagens consecutivas, como no pré-processamento do SISSA
    n_pedacos = len(tabela) // 512
    pedaco_com_ataque = atacante[: n_pedacos * 512].reshape(n_pedacos, 512).any(axis=1)
    linhas.append({'arquivo': arquivo, 'classe': classe, 'mensagens': len(tabela),
                   'mensagens_atacante': int(atacante.sum()), 'pedacos_512': n_pedacos,
                   'pedacos_com_ataque': int(pedaco_com_ataque.sum())})

resumo_rotulos = pd.DataFrame(linhas)
print(resumo_rotulos.to_string(index=False))
print('\nPor classe:')
print(resumo_rotulos.groupby('classe')[['mensagens', 'mensagens_atacante', 'pedacos_512', 'pedacos_com_ataque']].sum())

                         arquivo      classe  mensagens  mensagens_atacante  pedacos_512  pedacos_com_ataque
             benign_traffic.pcap      Normal    1108704                   0         2165                   0
             dos_noti_flood.pcap         DoS     973423              251860         1901                1901
fuzzy_sd_offer_rand_noti(1).pcap       Fuzzy    1113682                5243         2175                 758
fuzzy_sd_offer_rand_noti(2).pcap       Fuzzy     652722               36440         1274                1149
fuzzy_sd_offer_rand_noti(3).pcap       Fuzzy    1123430              511592         2194                2194
        mitm_multi_attacker.pcap  MITM_multi    1293930              171565         2527                2526
       mitm_single_attacker.pcap MITM_single    1131971              359373         2210                2207

Por classe:
             mensagens  mensagens_atacante  pedacos_512  pedacos_com_ataque
classe                                 

In [21]:
# 13) Geração das janelas no formato do SISSA, igualação das classes e divisão 80/20 (protocolo dos autores)
!pip install -q ruamel.yaml==0.17.21  # dependência do pacote preprocess do SISSA (leitura dos arquivos de configuração)

# Obtém a versão mais recente do repositório do grupo (código dos autores + módulos em projeto/kim)
if os.path.exists('/content/SISSA'):
    !git -C /content/SISSA pull -q
else:
    !git clone -q https://github.com/getuliojql/SISSA-CIN0114.git /content/SISSA
sys.path.insert(0, '/content/SISSA')              # raiz do repositório: acesso a preprocess/utils.py
sys.path.insert(0, '/content/SISSA/projeto/kim')  # módulos do grupo
from preprocess import utils
import gerar_janelas as gj

IPS_BENIGNOS = {f'172.18.0.{i}' for i in range(2, 11)}
CLASSES = ['Normal', 'DoS', 'Fuzzy', 'MITM_single', 'MITM_multi']  # a posição na lista é o rótulo
ARQUIVOS_POR_CLASSE = {
    'Normal': ['benign_traffic.parquet'],
    'DoS': ['dos_noti_flood.parquet'],
    'Fuzzy': ['fuzzy_sd_offer_rand_noti(1).parquet', 'fuzzy_sd_offer_rand_noti(2).parquet',
              'fuzzy_sd_offer_rand_noti(3).parquet'],
    'MITM_single': ['mitm_single_attacker.parquet'],
    'MITM_multi': ['mitm_multi_attacker.parquet'],
}
JANELAS = os.path.join(PASTA, 'janelas')

# Seleção dos pedaços de 512 mensagens, geração e sorteio das janelas, divisão 80/20 e gravação no Drive
inicio = time.time()
pedacos = gj.pedacos_por_classe(SAIDA, ARQUIVOS_POR_CLASSE, IPS_BENIGNOS)
X_tr, y_tr, X_val, y_val, contagens = gj.montar_dataset(pedacos, CLASSES, janelas_por_classe=2573, utils=utils)
gj.salvar(JANELAS, X_tr, y_tr, X_val, y_val)
print(contagens.to_string(index=False))
print(f'Treino: {X_tr.shape}, validação: {X_val.shape} ({time.time() - inicio:.0f} s)')

     classe  rotulo  pedacos_disponiveis  pedacos_usados  treino  validacao
     Normal       0                 2165            1524    2058        515
        DoS       1                 1901            1429    2058        515
      Fuzzy       2                 4101            1908    2058        515
MITM_single       3                 2207            1553    2058        515
 MITM_multi       4                 2526            1579    2058        515
Treino: (10290, 128, 25), validação: (2575, 128, 25) (102 s)


In [3]:
# 14) Preparação do treino no dataset do Kim: dependências, repositório, dados e arquivos de configuração
!nvidia-smi -L  # confirma a GPU alocada
!pip install -q concurrent-log-handler==0.9.25 ruamel.yaml==0.17.21 torchinfo==1.8.0

if os.path.exists('/content/SISSA'):
    !git -C /content/SISSA pull -q
else:
    !git clone -q https://github.com/getuliojql/SISSA-CIN0114.git /content/SISSA
%cd /content/SISSA

# Cópia das janelas do Drive para o disco local, de leitura mais rápida durante o treino
!mkdir -p data_kim && cp -r "{PASTA}/janelas/train" "{PASTA}/janelas/val" data_kim/

# Ajuste do config/basic.yml para o dataset do Kim: diretórios de dados, classes e dimensões das janelas
from ruamel.yaml import YAML
yaml = YAML()
CLASSES = ['Normal', 'DoS', 'Fuzzy', 'MITM_single', 'MITM_multi']
basic = yaml.load(open('config/basic.yml'))
basic['Data']['train_dir'] = 'data_kim/train'
basic['Data']['val_dir'] = 'data_kim/val'
basic['Preprocessor']['class_names'] = CLASSES
basic['Preprocessor']['n_classes'] = len(CLASSES)
basic['Preprocessor']['window_height'] = 128
basic['Preprocessor']['window_width'] = 25
yaml.dump(basic, open('config/basic.yml', 'w'))

GPU 0: Tesla T4 (UUID: GPU-26474cf6-7b49-71b8-26a6-65f64dee4c09)
/content/SISSA


In [4]:
# 15) Treino e avaliação dos modelos no dataset do Kim, com os mesmos hiperparâmetros da reprodução
import importlib.util
import matplotlib.pyplot as plt
import initialize, sissautils
from models import SISSA_MODELS
from train import train_pattern

# Importação do test.py do repositório pelo caminho do arquivo, para evitar conflito com o módulo 'test' do Python
spec = importlib.util.spec_from_file_location('local_test', 'test.py')
local_test = importlib.util.module_from_spec(spec)
spec.loader.exec_module(local_test)
test_pattern = local_test.test_pattern
initialize.init()

MODELOS = ['SISSA-L-A', 'SISSA-L']
EPOCAS = 150  # conforme a Seção V-B do artigo

def melhor_checkpoint(pasta):
    """Retorna o checkpoint de maior acurácia de validação (mesmo critério do notebook de reprodução)."""
    pts = [f for f in os.listdir(pasta) if f.endswith('.pt')]
    pts.sort(key=lambda f: (float(f[:-3].split('val-acc-')[1]), int(f.split('_')[0].split('-')[1])))
    return os.path.join(pasta, pts[-1])

for modelo in MODELOS:
    # Configuração do modelo: nome da execução, número de épocas e conjunto de teste
    caminho_cfg = f'config/{modelo}.yml'
    cfg = yaml.load(open(caminho_cfg))
    cfg['Model']['name'] = f'{modelo}_kim'
    cfg['Train']['epochs'] = EPOCAS
    cfg['Test']['test_dir'] = 'data_kim/val'  # teste no conjunto de validação, como no artigo
    yaml.dump(cfg, open(caminho_cfg, 'w'))

    # Treino, escolha do melhor checkpoint e teste
    cfg = sissautils.update_model_config(model_config_path=caminho_cfg)
    inicio = time.time()
    rede = SISSA_MODELS[cfg['Model']['type']](**cfg['Params'])
    hparams = sissautils.hyperparams(model_config=cfg, model=rede)
    train_pattern(**hparams)
    test_pattern(model_config_path=caminho_cfg, weights_path=melhor_checkpoint(hparams['weights_dir']))
    plt.close('all')
    print(f'{modelo}: {(time.time() - inicio) / 60:.1f} min')

# Cópia dos pesos, logs de treino e resultados de teste para o Drive
!mkdir -p "{PASTA}/resultados_treino"
!cp -r results/weights results/train_log results/test_result "{PASTA}/resultados_treino/"

[to_dataloader][CRITICAL] >>>>> Sending data to dataloader...
CRITICAL:Train:Sending data to dataloader...
[to_dataloader][INFO] >>>>> Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
INFO:Train:Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/train/labels.npy, shape: (10290,)
INFO:Train:Loaded label from data_kim/train/labels.npy, shape: (10290,)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[to_dataloader][CRITICAL] >>>>> Data sent to dataloader.
CRITICAL:Train:Data sent 

SISSA-L-A: 13.2 min


[to_dataloader][INFO] >>>>> Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
INFO:Train:Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/train/labels.npy, shape: (10290,)
INFO:Train:Loaded label from data_kim/train/labels.npy, shape: (10290,)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[to_dataloader][CRITICAL] >>>>> Data sent to dataloader.
CRITICAL:Train:Data sent to dataloader.
[to_dataloader][CRITICAL] >>>>> Sending data to dataloader...
CRITICAL:Train:Sending data to

SISSA-L: 10.9 min


In [5]:
# 16) Precisão, recall e F1 por classe no conjunto de teste, a partir das previsões salvas pelo test.py
from sklearn.metrics import classification_report
for modelo in MODELOS:
    pasta = f'results/test_result/{modelo}_kim'
    y_true, y_pred = np.load(f'{pasta}/true.npy'), np.load(f'{pasta}/pred.npy')
    print(f'\n===== {modelo} =====')
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))


===== SISSA-L-A =====
              precision    recall  f1-score   support

      Normal     0.5590    0.8466    0.6734       515
         DoS     0.9981    0.9942    0.9961       515
       Fuzzy     0.6827    0.3301    0.4450       515
 MITM_single     0.9942    1.0000    0.9971       515
  MITM_multi     0.9981    0.9981    0.9981       515

    accuracy                         0.8338      2575
   macro avg     0.8464    0.8338    0.8219      2575
weighted avg     0.8464    0.8338    0.8219      2575


===== SISSA-L =====
              precision    recall  f1-score   support

      Normal     0.5623    0.8505    0.6770       515
         DoS     0.9903    0.9883    0.9893       515
       Fuzzy     0.6908    0.3340    0.4503       515
 MITM_single     0.9884    0.9903    0.9893       515
  MITM_multi     0.9942    0.9981    0.9961       515

    accuracy                         0.8322      2575
   macro avg     0.8452    0.8322    0.8204      2575
weighted avg     0.8452    0.8322

### Análise complementar: demais variantes do SISSA no dataset do Kim

As células anteriores avaliam o sistema proposto no artigo (SISSA-L-A) e a mesma arquitetura sem o bloco de atenção (SISSA-L). Esta seção treina as quatro variantes restantes (SISSA-C, SISSA-C-A, SISSA-R e SISSA-R-A), com os mesmos dados e hiperparâmetros, para repetir no dataset do Kim a comparação entre arquiteturas (CNN, RNN e LSTM, com e sem atenção) feita no artigo.

In [6]:
# 17) Análise complementar: treino e avaliação das variantes CNN e RNN, com e sem atenção, no dataset do Kim
MODELOS_COMPLEMENTARES = ['SISSA-C', 'SISSA-C-A', 'SISSA-R', 'SISSA-R-A']

for modelo in MODELOS_COMPLEMENTARES:
    # Configuração do modelo: nome da execução, número de épocas e conjunto de teste
    caminho_cfg = f'config/{modelo}.yml'
    cfg = yaml.load(open(caminho_cfg))
    cfg['Model']['name'] = f'{modelo}_kim'
    cfg['Train']['epochs'] = EPOCAS
    cfg['Test']['test_dir'] = 'data_kim/val'  # teste no conjunto de validação, como no artigo
    yaml.dump(cfg, open(caminho_cfg, 'w'))

    # Treino, escolha do melhor checkpoint e teste
    cfg = sissautils.update_model_config(model_config_path=caminho_cfg)
    inicio = time.time()
    rede = SISSA_MODELS[cfg['Model']['type']](**cfg['Params'])
    hparams = sissautils.hyperparams(model_config=cfg, model=rede)
    train_pattern(**hparams)
    test_pattern(model_config_path=caminho_cfg, weights_path=melhor_checkpoint(hparams['weights_dir']))
    plt.close('all')
    print(f'{modelo}: {(time.time() - inicio) / 60:.1f} min')

# Cópia dos pesos, logs de treino e resultados de teste para o Drive
!cp -r results/weights results/train_log results/test_result "{PASTA}/resultados_treino/"

[to_dataloader][CRITICAL] >>>>> Sending data to dataloader...
CRITICAL:Train:Sending data to dataloader...
[to_dataloader][INFO] >>>>> Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
INFO:Train:Loaded data from data_kim/train/data.npy, shape: (10290, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/train/labels.npy, shape: (10290,)
INFO:Train:Loaded label from data_kim/train/labels.npy, shape: (10290,)
[to_dataloader][CRITICAL] >>>>> Data sent to dataloader.
CRITICAL:Train:Data sent to dataloader.
[to_dataloader][CRITICAL] >>>>> Sending data to dataloader...
CRITICAL:Train:Sending data to dataloader...
[to_dataloader][INFO] >>>>> Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
INFO:Train:Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/val/labels.npy, shape: (2575,)
INFO:Train:Loaded label from data_kim/val/labels.npy, shape: (2575,)
[to_dataloader][CRITICAL] >>>>> Data 

SISSA-C: 3.7 min


CRITICAL:Train:Data sent to dataloader.
[train_pattern][CRITICAL] >>>>> Start training...
CRITICAL:Train:Start training...
[train_pattern][INFO] >>>>> Epoch:    1 | Train acc: 0.3397, Val acc: 0.4474
INFO:Train:Epoch:    1 | Train acc: 0.3397, Val acc: 0.4474
[train_pattern][INFO] >>>>> Epoch:    2 | Train acc: 0.6046, Val acc: 0.6928
INFO:Train:Epoch:    2 | Train acc: 0.6046, Val acc: 0.6928
[train_pattern][INFO] >>>>> Epoch:    3 | Train acc: 0.7332, Val acc: 0.7390
INFO:Train:Epoch:    3 | Train acc: 0.7332, Val acc: 0.7390
[train_pattern][INFO] >>>>> Epoch:    4 | Train acc: 0.7588, Val acc: 0.7623
INFO:Train:Epoch:    4 | Train acc: 0.7588, Val acc: 0.7623
[train_pattern][INFO] >>>>> Epoch:    5 | Train acc: 0.7695, Val acc: 0.7751
INFO:Train:Epoch:    5 | Train acc: 0.7695, Val acc: 0.7751
[train_pattern][INFO] >>>>> Epoch:    6 | Train acc: 0.7770, Val acc: 0.7755
INFO:Train:Epoch:    6 | Train acc: 0.7770, Val acc: 0.7755
[train_pattern][INFO] >>>>> Epoch:    7 | Train acc: 0.

SISSA-C-A: 3.8 min


[to_dataloader][INFO] >>>>> Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
INFO:Train:Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/val/labels.npy, shape: (2575,)
INFO:Train:Loaded label from data_kim/val/labels.npy, shape: (2575,)
[to_dataloader][CRITICAL] >>>>> Data sent to dataloader.
CRITICAL:Train:Data sent to dataloader.
[train_pattern][CRITICAL] >>>>> Start training...
CRITICAL:Train:Start training...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[train_pattern][INFO]

SISSA-R: 3.0 min


[to_dataloader][INFO] >>>>> Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
INFO:Train:Loaded data from data_kim/val/data.npy, shape: (2575, 128, 25)
[to_dataloader][INFO] >>>>> Loaded label from data_kim/val/labels.npy, shape: (2575,)
INFO:Train:Loaded label from data_kim/val/labels.npy, shape: (2575,)
[to_dataloader][CRITICAL] >>>>> Data sent to dataloader.
CRITICAL:Train:Data sent to dataloader.
[train_pattern][CRITICAL] >>>>> Start training...
CRITICAL:Train:Start training...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[train_pattern][INFO]

SISSA-R-A: 4.0 min


In [7]:
# 18) Tabela-resumo dos seis modelos no dataset do Kim: acurácia, F1 macro e F1 por classe
from sklearn.metrics import accuracy_score, f1_score

ORDEM = ['SISSA-C', 'SISSA-C-A', 'SISSA-R', 'SISSA-R-A', 'SISSA-L', 'SISSA-L-A']  # ordem das tabelas do artigo
linhas = []
for modelo in ORDEM:
    pasta = f'results/test_result/{modelo}_kim'
    y_true, y_pred = np.load(f'{pasta}/true.npy'), np.load(f'{pasta}/pred.npy')
    linha = {'modelo': modelo,
             'acuracia': accuracy_score(y_true, y_pred),
             'f1_macro': f1_score(y_true, y_pred, average='macro')}
    linha.update({f'f1_{c}': v for c, v in zip(CLASSES, f1_score(y_true, y_pred, average=None))})
    linhas.append(linha)

tabela_kim = pd.DataFrame(linhas).round(4)
print(tabela_kim.to_string(index=False))
tabela_kim.to_csv(f'{PASTA}/resultados_treino/tabela_kim.csv', index=False)

   modelo  acuracia  f1_macro  f1_Normal  f1_DoS  f1_Fuzzy  f1_MITM_single  f1_MITM_multi
  SISSA-C    0.8085    0.8074     0.6007  0.9612    0.5207          0.9739         0.9804
SISSA-C-A    0.7930    0.7845     0.6106  0.9574    0.4029          0.9718         0.9795
  SISSA-R    0.8151    0.8049     0.6378  0.9854    0.4177          0.9884         0.9951
SISSA-R-A    0.8136    0.8082     0.6170  0.9835    0.4577          0.9864         0.9961
  SISSA-L    0.8322    0.8204     0.6770  0.9893    0.4503          0.9893         0.9961
SISSA-L-A    0.8338    0.8219     0.6734  0.9961    0.4450          0.9971         0.9981
